# Regression with a feed-forward network: diabetes progression

In this notebook we predict a **real number** (how much a patient's diabetes progresses
in one year) from 10 medical measurements. This is the same idea as the XOR regression
example (`S41_XOR_regression.py`), but on a real dataset:

- the model ends with a linear layer with **no activation**, so it can output any real number;
- training uses **mean squared error (MSE)**.

Steps: load and explore the data → split into train/test → standardize → build the
model → train → evaluate on the test set.

In [ ]:
# ── 0. Imports ─────────────────────────────────────────────────────────────
import pandas as pd  # tables (DataFrame) for exploring the data
import seaborn as sns  # quick statistical plots
import torch
from sklearn.datasets import load_diabetes  # built-in example dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

In [ ]:
# ── 1. Data ────────────────────────────────────────────────────────────────
# The diabetes dataset: 442 patients, 10 features each (age, sex, bmi, bp,
# and six blood measurements s1–s6). The target is a number measuring how
# much the disease has progressed one year later.
#
# scaled=False gives the raw values (e.g. age in years), so we can see the
# real units and do the scaling ourselves later (step 4).
diabetes = load_diabetes(scaled=False)

print(diabetes.DESCR)  # full description of the dataset

In [ ]:
# Put the features and the target into one table so they are easy to explore.
# Each row is one patient; the last column ("target") is what we predict.
df = pd.DataFrame(data=diabetes.data, columns=diabetes.feature_names)
df["target"] = diabetes.target
df

In [ ]:
# ── 2. Explore ─────────────────────────────────────────────────────────────
# Histogram of every column: shows the range and shape of each variable.
# Note how different the ranges are (e.g. age vs. s1). We fix this in step 4.
df.hist(bins=20, figsize=(12, 10))

In [ ]:
# Correlation between every pair of columns, from -1 to +1.
#   +1 -> increase together;  -1 -> one goes up, the other goes down;
#    0 -> no linear relationship.
# Look at the "target" row: which features relate most to the target?
corr_matrix = df.corr()
corr_matrix

In [ ]:
# The same correlation matrix as a heatmap: easier to spot strong values.
sns.heatmap(corr_matrix)

In [ ]:
# Boxplot of the features (all columns except the target) on one axis.
# The features have very different scales, which can slow down training.
df.iloc[:, :-1].boxplot()

In [ ]:
# ── 3. Inputs, targets and train/test split ────────────────────────────────
# Split the table into inputs and target, as NumPy arrays.
#   _X : all columns except the last -> shape (442, 10)
#   _Y : only the last column        -> shape (442, 1)
# Using -1: (not -1) keeps _Y 2D, which matches the model's output shape.
# The leading "_" marks raw (not yet scaled) data.
_X = df.iloc[:, :-1].values
_Y = df.iloc[:, -1:].values
print(_X.shape)
print(_Y.shape)

In [ ]:
# Keep 30% of the patients aside as a test set. The model never sees them
# during training, so they tell us how well it works on new patients.
# random_state=0 makes the split the same on every run.
_X_train, _X_test, _Y_train, _Y_test = train_test_split(
    _X, _Y, test_size=0.3, random_state=0
)
print(_X_train.shape)
print(_X_test.shape)
print(_Y_train.shape)
print(_Y_test.shape)

In [ ]:
# ── 4. Standardize ─────────────────────────────────────────────────────────
# StandardScaler turns each column into: (value - mean) / std
# so every column has mean 0 and standard deviation 1. With all inputs on
# a similar scale, gradient descent trains faster and more reliably.
#
# fit_transform on the TRAIN set: learn mean/std from the training data.
# transform on the TEST set  : reuse the training mean/std.
# (Fitting on the test set would leak information about it into training.)
scX = StandardScaler()
X_train = scX.fit_transform(_X_train)
X_test = scX.transform(_X_test)

# The target is standardized too, with its own scaler. The model then
# predicts values in standard units; scY.inverse_transform() converts them
# back to the original target scale if needed.
scY = StandardScaler()
Y_train = scY.fit_transform(_Y_train)
Y_test = scY.transform(_Y_test)

In [ ]:
# Compare with the earlier boxplot: all features are now centered at 0
# with a similar spread.
pd.DataFrame(data=X_train).boxplot()

In [ ]:
# PyTorch works with tensors, not NumPy arrays.
# .float() converts to float32: NumPy uses float64 by default, but the
# model's weights are float32, and the two types cannot be mixed.
X_train_pt = torch.from_numpy(X_train).float()
X_test_pt = torch.from_numpy(X_test).float()
Y_train_pt = torch.from_numpy(Y_train).float()
Y_test_pt = torch.from_numpy(Y_test).float()

In [ ]:
# ── 5. Model ───────────────────────────────────────────────────────────────
# Read the input and output sizes from the data instead of hard-coding them.
num_features = _X.shape[1]  # 10 inputs per patient
num_outputs = _Y.shape[1]  # 1 predicted value
print(num_features, num_outputs)

In [ ]:
# Three hidden layers that get narrower (24 -> 12 -> 6), each followed by
# ReLU. ReLU(x) = max(0, x) is the usual non-linearity for hidden layers;
# without it, the stacked layers would collapse into one linear model.
#
# The output layer has no activation, so the model can output any real
# number. That is normal for regression.
model = nn.Sequential(
    nn.Linear(num_features, 24),  # hidden layer 1: inputs -> 24 neurons
    nn.ReLU(),
    nn.Linear(24, 12),  # hidden layer 2: 24 -> 12 neurons
    nn.ReLU(),
    nn.Linear(12, 6),  # hidden layer 3: 12 -> 6 neurons
    nn.ReLU(),
    nn.Linear(6, num_outputs),  # output layer: no activation (real-valued output)
)

In [ ]:
# Count the trainable numbers (weights + biases) in the model.
# A Linear(a, b) layer has a*b weights and b biases:
#   10*24 + 24 = 264
#   24*12 + 12 = 300
#   12*6  + 6  =  78
#   6*1   + 1  =   7
#   total      = 649
pytorch_total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {pytorch_total_params}")

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────────
# One epoch = one pass over the whole training set. Here every epoch uses
# all training samples at once (full-batch gradient descent).
n_epochs = 2000

# Adam is a variant of gradient descent that adapts the step size for
# each weight. Try changing the learning rate (lr) to see its effect:
#   too small -> slow training; too large -> loss may jump around.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# MSELoss = mean of (prediction - target)^2 over all samples.
# Since the target is standardized, a loss of 1.0 is about as good as
# always predicting the average; lower is better.
loss_fn = nn.MSELoss()

loss_arr = []  # training loss of each epoch, for plotting later
for epoch in range(n_epochs):

    model.train()  # training mode (matters for layers like Dropout; none here)
    
    Y_pred = model(X_train_pt)  # forward pass
    loss = loss_fn(Y_pred, Y_train_pt)  # compare predictions with targets
    
    optimizer.zero_grad()  # clear gradients from the previous step
    loss.backward()  # backpropagation: compute gradients
    optimizer.step()  # update weights

    epoch_train_loss = loss.item()  # .item() turns a 1-element tensor into a Python float
    loss_arr.append(epoch_train_loss)

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}: train loss = {epoch_train_loss:.4f}")


In [ ]:
# Training loss over the epochs. It should drop quickly at first and then
# level off.
sns.lineplot(loss_arr)

In [ ]:
# ── 7. Evaluate ────────────────────────────────────────────────────────────
# Compute the loss on the test set, which the model has never seen.
# no_grad() turns off gradient tracking: we are only evaluating, not training.
#
# If the test loss is much higher than the final training loss, the model
# has overfit: it memorized the training patients instead of learning a
# pattern that carries over to new ones.
with torch.no_grad():
    test_pred = model(X_test_pt)
    final_loss = loss_fn(test_pred, Y_test_pt)

print(final_loss)